## Project Data Science Lab 2 

In [24]:
import pandas as pd
import re
import os
import gensim
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
import numpy as np
import nltk

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Preprocessing

In [25]:
# Load the stopwords
stop_words = set(stopwords.words('english'))

# Function to preprocess the text without using NLTK
def preprocess_text(text):
    # Remove newline characters and replace with a space
    text = text.replace('\n', ' ').replace('\r', ' ')
    # Convert to lowercase
    text = text.lower()
    # Remove special characters and digits
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\d', ' ', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenize the text (split by spaces)
    words = text.split()
    # Remove stopwords using the custom stopwords list
    words = [word for word in words if word not in stop_words]
    return words

In [26]:
# Apply preprocessing to the course details
file_path = '../annotated_course_details.xlsx'
df_courses = pd.read_excel(file_path)
# df['preprocessed_course_detail'] = df['course detail description'].apply(preprocess_text)
# df.head()
df_courses['preprocessed_course_detail'] = df_courses['course detail description'].apply(preprocess_text)
df_courses['preprocessed_task_phrases'] = df_courses['task_phrases'].apply(preprocess_text)
df_courses['preprocessed_skill_phrases'] = df_courses['skill_phrases'].apply(preprocess_text)
# df
# Combine annotated phrases with preprocessed course details
df_courses['combined_text_courses'] = (
    df_courses['preprocessed_course_detail'].apply(lambda x: " ".join(x)) + " " + 
    df_courses['preprocessed_task_phrases'].apply(lambda x: " ".join(x)) + " " + 
    df_courses['preprocessed_skill_phrases'].apply(lambda x: " ".join(x)))

In [27]:
# Load the skill roles dataset
df_skill_roles = pd.read_excel('../annotated_skill_roles.xlsx')

# Apply preprocessing to the mission, main tasks, and key skills columns
df_skill_roles['preprocessed_mission'] = df_skill_roles['mission'].apply(preprocess_text)
df_skill_roles['preprocessed_main_tasks'] = df_skill_roles['main_tasks'].apply(preprocess_text)
df_skill_roles['preprocessed_key_skills'] = df_skill_roles['key_skills'].apply(preprocess_text)
df_skill_roles['preprocessed_task_phrases'] = df_skill_roles['unique_task_phrases'].apply(preprocess_text)
df_skill_roles['preprocessed_skill_phrases'] = df_skill_roles['unique_skill_phrases'].apply(preprocess_text)

# Combine the preprocessed text and annotated phrases into a single column
df_skill_roles['combined_text_skill_roles'] = (
    df_skill_roles['preprocessed_mission'].apply(lambda x: " ".join(x)) + " " +
    df_skill_roles['preprocessed_main_tasks'].apply(lambda x: " ".join(x)) + " " +
    df_skill_roles['preprocessed_key_skills'].apply(lambda x: " ".join(x)) + " " +
    df_skill_roles['preprocessed_task_phrases'].apply(lambda x: " ".join(x)) + " " +
    df_skill_roles['preprocessed_skill_phrases'].apply(lambda x: " ".join(x))
)

In [28]:
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.metrics.pairwise import cosine_similarity
import time

# Load your datasets
# Extract the combined preprocessed text
courses_text = df_courses['combined_text_courses'].tolist()
skill_roles_text = df_skill_roles['combined_text_skill_roles'].tolist()

# Tag the documents (necessary for Doc2Vec)
tagged_courses = [TaggedDocument(words=text.split(), tags=[f'COURSE_{i}']) for i, text in enumerate(courses_text)]
tagged_skill_roles = [TaggedDocument(words=text.split(), tags=[f'SKILL_ROLE_{i}']) for i, text in enumerate(skill_roles_text)]
tagged_documents = tagged_courses + tagged_skill_roles

# Hyperparameter configurations to test
configs = [
    {"vector_size": 50, "window": 7, "epochs": 30, "dm": 0},
    {"vector_size": 50, "window": 10, "epochs": 30, "dm": 0},
    {"vector_size": 50, "window": 15, "epochs": 30, "dm": 0},
    {"vector_size": 50, "window": 20, "epochs": 30, "dm": 0},
    {"vector_size": 300, "window": 7, "epochs": 30, "dm": 0},
    {"vector_size": 300, "window": 10, "epochs": 30, "dm": 0},
    {"vector_size": 300, "window": 15, "epochs": 30, "dm": 0},
    {"vector_size": 300, "window": 20, "epochs": 30, "dm": 0},
]

# Store results
results = []

for config in configs:
    start_time = time.time()
    
    # Initialize the Doc2Vec model with the current configuration
    model = Doc2Vec(
        vector_size=config["vector_size"],
        window=config["window"],
        min_count=2,
        workers=4,
        epochs=config["epochs"],
        dm=config["dm"]
    )

    # Build the vocabulary and train the model
    model.build_vocab(tagged_documents)
    model.train(tagged_documents, total_examples=model.corpus_count, epochs=model.epochs)
    
    # Extract vectors for courses and skill roles
    course_vectors = [model.dv[f'COURSE_{i}'] for i in range(len(courses_text))]
    skill_role_vectors = [model.dv[f'SKILL_ROLE_{i}'] for i in range(len(skill_roles_text))]
    
    # Calculate the cosine similarity between each course and each skill role
    similarity_matrix = cosine_similarity(course_vectors, skill_role_vectors)
    
    # Calculate metrics
    avg_similarity = similarity_matrix.mean()
    max_similarity = similarity_matrix.max()  # Get the highest similarity value
    negative_scores_percentage = (similarity_matrix < 0).mean() * 100
    training_time = time.time() - start_time
    
    # Store results
    results.append({
        "Vector Size": config["vector_size"],
        "Window": config["window"],
        "Epochs": config["epochs"],
        "Distributed Memory": "Yes" if config["dm"] == 1 else "No",
        "Mean Similarity": avg_similarity,
        "Highest Similarity": max_similarity,
        "Negative Scores (%)": negative_scores_percentage,
        "Training Time (s)": training_time
    })

# Convert the results to a DataFrame for easier analysis
results_df = pd.DataFrame(results)

# Display the results
results_df

# Save the results to an Excel file
results_df.to_excel('doc2vec_hyperparameter_comparison.xlsx', index=False)


In [29]:
results_df

,Vector Size,Window,Epochs,Distributed Memory,Mean Similarity,Highest Similarity,Negative Scores (%),Training Time (s)
0,50,7,30,No,0.319924,0.808090,0.192482,4.454995
1,50,10,30,No,0.317344,0.785944,0.203804,4.621001
2,50,15,30,No,0.316442,0.810063,0.147192,3.789004
3,50,20,30,No,0.317617,0.783367,0.192482,3.100996
4,300,7,30,No,0.314506,0.804193,0.135870,3.885003
5,300,10,30,No,0.313910,0.809645,0.135870,5.375848
6,300,15,30,No,0.315537,0.816040,0.124547,4.980412
7,300,20,30,No,0.315353,0.818726,0.147192,5.363521


In [49]:
# Load your datasets
# Assuming df_courses and df_skill_roles contain the relevant titles and text
courses_text = df_courses['combined_text_courses'].tolist()
course_titles = df_courses['course title'].tolist()  
skill_roles_text = df_skill_roles['combined_text_skill_roles'].tolist()
skill_role_names = df_skill_roles['profile_title'].tolist() 

# Tag the documents (necessary for Doc2Vec)
tagged_courses = [TaggedDocument(words=text.split(), tags=[f'COURSE_{i}']) for i, text in enumerate(courses_text)]
tagged_skill_roles = [TaggedDocument(words=text.split(), tags=[f'SKILL_ROLE_{i}']) for i, text in enumerate(skill_roles_text)]
tagged_documents = tagged_courses + tagged_skill_roles

# Hyperparameter configuration (you can adjust this or iterate through multiple configurations as before)
config = {"vector_size": 300, "window": 20, "epochs": 30, "dm": 0}

# Initialize the Doc2Vec model with the current configuration
model = Doc2Vec(
    vector_size=config["vector_size"],
    window=config["window"],
    min_count=2,
    workers=4,
    epochs=config["epochs"],
    dm=config["dm"]
)

# Build the vocabulary and train the model
model.build_vocab(tagged_documents)
model.train(tagged_documents, total_examples=model.corpus_count, epochs=model.epochs)

# Extract vectors for courses and skill roles
course_vectors = [model.dv[f'COURSE_{i}'] for i in range(len(courses_text))]
skill_role_vectors = [model.dv[f'SKILL_ROLE_{i}'] for i in range(len(skill_roles_text))]

# Calculate the cosine similarity between each course and each skill role
similarity_matrix = cosine_similarity(course_vectors, skill_role_vectors)

# Create a DataFrame to store the similarity scores, with course titles and skill role names as the index and columns
similarity_df = pd.DataFrame(
    similarity_matrix,
    # index=course_titles,  # Use course titles as the index
    columns=skill_role_names  # Use skill role names as the columns
)

# Display the DataFrame
similarity_df

# Add the course titles as a separate column for clarity
similarity_df['Course Title'] = course_titles

# Move 'Course Title' to the front of the DataFrame
similarity_df = similarity_df[['Course Title'] + similarity_df.columns[:-1].tolist()]

# Save the DataFrame to an Excel file
similarity_df.to_excel('course_skill_role_similarity_scores.xlsx')


In [50]:
# Define thresholds for relevance
relevance_threshold = 0.50
potential_threshold = 0.40

# Create a DataFrame for the similarity scores
similarity_df = pd.DataFrame(
    similarity_matrix,
    # index=course_titles,  # Use course titles as the index
    columns=skill_role_names  # Use skill role names as the columns
)

# Convert similarity scores to 'Relevant', 'Potentially Relevant', or 'Not Relevant'
def categorize_similarity(score):
    if score > relevance_threshold:
        return "Relevant"
    elif score > potential_threshold:
        return "Potentially Relevant"
    else:
        return "Not Relevant"

# Apply the categorization function to the similarity matrix
categorized_df = similarity_df.applymap(categorize_similarity)

# Add the course titles as a separate column for clarity
categorized_df['Course Title'] = course_titles

# Move 'Course Title' to the front of the DataFrame
categorized_df = categorized_df[['Course Title'] + categorized_df.columns[:-1].tolist()]

# Display the DataFrame
categorized_df


C:\Users\ASUS\AppData\Local\Temp\ipykernel_13968\4125020828.py:22: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  categorized_df = similarity_df.applymap(categorize_similarity)


,Course Title,Chief Information Security Officer (CISO),Cyber Incident Responder,"Cyber Legal, Policy & Compliance Officer",Cyber Threat Intelligence Specialist,Cybersecurity Architect,Cybersecurity Auditor,Cybersecurity Educator,Cybersecurity Implementer,Cybersecurity Researcher,Cybersecurity Risk Manager,Digital Forensics Investigator,Penetration Tester
0,Introduction to Data Science Tools,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant
1,Research Methods and Scientific Ethics,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant
2,Cyber Security Basics,Not Relevant,Relevant,Potentially Relevant,Not Relevant,Potentially Relevant,Not Relevant,Potentially Relevant,Relevant,Potentially Relevant,Potentially Relevant,Not Relevant,Not Relevant
3,Computer Networks and Security,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant
4,Operating Systems and System Programming,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant
...,...,...,...,...,...,...,...,...,...,...,...,...,...
731,Digital Forensics,Not Relevant,Potentially Relevant,Not Relevant,Not Relevant,Not Relevant,Potentially Relevant,Not Relevant,Not Relevant,Not Relevant,Not Relevant,Relevant,Not Relevant
732,Security Audit in Digital Systems and Networks,Potentially Relevant,Relevant,Relevant,Potentially Relevant,Potentially Relevant,Relevant,Not Relevant,Relevant,Potentially Relevant,Potentially Relevant,Relevant,Relevant
733,Software and Application Security,Not Relevant,Potentially Relevant,Potentially Relevant,Not Relevant,Potentially Relevant,Potentially Relevant,Not Relevant,Relevant,Not Relevant,Potentially Relevant,Not Relevant,Potentially Relevant
734,Security Operations and Incident Management,Relevant,Relevant,Potentially Relevant,Relevant,Not Relevant,Potentially Relevant,Potentially Relevant,Potentially Relevant,Not Relevant,Relevant,Relevant,Not Relevant


In [56]:
# Initialize a list to store the relevant courses and their top skill profiles
relevant_courses = []

# Iterate over each row in categorized_df using the integer index
for index, row in categorized_df.iterrows():
    course_title = row['Course Title']
    
    # Collect relevant and potentially relevant skill profiles
    relevant_skills = []
    for skill_profile in similarity_df.columns:
        if row[skill_profile] in ['Relevant', 'Potentially Relevant']:
            similarity_score = round(similarity_df.at[index, skill_profile], 2)
            relevant_skills.append((skill_profile, similarity_score, row[skill_profile]))
    
    # Sort by similarity score in descending order and take the top 3
    relevant_skills = sorted(relevant_skills, key=lambda x: x[1], reverse=True)[:3]
    
    # If there are relevant skills, add them to the list
    if relevant_skills:
        for skill_profile, score, category in relevant_skills:
            relevant_courses.append({
                'Course Title': course_title,
                'Relevant Skill Profile': skill_profile,
                'Category': category,
                'Similarity Score': score
            })
    else:
        # If no relevant or potentially relevant skills, add a record indicating no relevant skills
        relevant_courses.append({
            'Course Title': course_title,
            'Relevant Skill Profile': 'No skill roles relevant to this course',
            'Category': None,
            'Similarity Score': None
        })

# Convert the list to a DataFrame
relevant_courses_df = pd.DataFrame(relevant_courses)

# Display the DataFrame
relevant_courses_df

,Course Title,Relevant Skill Profile,Category,Similarity Score
0,Introduction to Data Science Tools,No skill roles relevant to this course,None,NaN
1,Research Methods and Scientific Ethics,No skill roles relevant to this course,None,NaN
2,Cyber Security Basics,Cybersecurity Implementer,Relevant,0.51
3,Cyber Security Basics,Cyber Incident Responder,Relevant,0.50
4,Cyber Security Basics,"Cyber Legal, Policy & Compliance Officer",Potentially Relevant,0.47
...,...,...,...,...
1368,Security Operations and Incident Management,Cyber Incident Responder,Relevant,0.58
1369,Security Operations and Incident Management,Cyber Threat Intelligence Specialist,Relevant,0.55
1370,Security Operations and Incident Management,Cybersecurity Risk Manager,Relevant,0.55
1371,"Technology, Economy and Society",Cybersecurity Researcher,Potentially Relevant,0.44


In [59]:
# Initialize a list to store courses with no relevant skill roles
no_relevant_courses = []

# Iterate over each row in categorized_df using the integer index
for index, row in categorized_df.iterrows():
    course_title = row['Course Title']
    
    # Check if all skill profiles for this course are marked as 'Not Relevant'
    if all(row[skill_profile] == 'Not Relevant' for skill_profile in similarity_df.columns):
        no_relevant_courses.append(course_title)

# Convert the list to a DataFrame
no_relevant_courses_df = pd.DataFrame(no_relevant_courses, columns=['Course Title'])

# Display the DataFrame and the count of such courses
print(f"Number of courses with no relevant skill roles: {len(no_relevant_courses_df)}")
no_relevant_courses_df

Number of courses with no relevant skill roles: 312


,Course Title
0,Introduction to Data Science Tools
1,Research Methods and Scientific Ethics
2,Computer Networks and Security
3,Operating Systems and System Programming
4,Turkish
...,...
307,Algorithms and Data Structures
308,Applied Mathematics Complements
309,Introduction to Computer Networks
310,Introduction to Design Thinking


In [60]:
# Save the DataFrame to an Excel file if needed
no_relevant_courses_df.to_excel('courses_with_no_relevant_skill_roles.xlsx', index=False)

In [58]:
# Initialize counters for alignment
aligned_courses_count = 0
non_aligned_courses_count = 0

# Iterate over each row in categorized_df using the integer index
for index, row in categorized_df.iterrows():
    # Check if any skill profile is marked as 'Relevant' or 'Potentially Relevant'
    if any(row[skill_profile] in ['Relevant', 'Potentially Relevant'] for skill_profile in similarity_df.columns):
        aligned_courses_count += 1
    else:
        non_aligned_courses_count += 1

# Output the results
print(f"Number of courses aligned with skill roles: {aligned_courses_count}")
print(f"Number of courses not aligned with any skill roles: {non_aligned_courses_count}")

Number of courses aligned with skill roles: 424
Number of courses not aligned with any skill roles: 312
